In [4]:


def is_variable(x):
    return isinstance(x, str) and x.startswith('?')


def occurs(var, expr, subst):
    expr = apply_subst(expr, subst)

    if var == expr:
        return True

    if isinstance(expr, tuple):
        return any(occurs(var, x, subst) for x in expr)

    return False


def apply_subst(expr, subst):
    if is_variable(expr) and expr in subst:
        return apply_subst(subst[expr], subst)

    if isinstance(expr, tuple):
        return tuple(apply_subst(x, subst) for x in expr)

    return expr

def unify(x, y, subst=None):

    if subst is None:
        subst = {}

    x = apply_subst(x, subst)
    y = apply_subst(y, subst)

    if x == y:
        return subst

    if is_variable(x):
        if occurs(x, y, subst):
            return None
        subst[x] = y
        return subst

    if is_variable(y):
        if occurs(y, x, subst):
            return None
        subst[y] = x
        return subst

    if isinstance(x, tuple) and isinstance(y, tuple):

        if x[0] != y[0]:
            return None
        if len(x) != len(y):
            return None

        for i in range(1, len(x)):
            subst = unify(x[i], y[i], subst)

            if subst is None:
                return None

        return subst

    return None

facts = [
    ("American", "Robert"),
    ("Enemy", "A", "America"),
    ("Missile", "M1"),
    ("Owns", "A", "M1")
]

rules = [
    ([("Missile", "?x")],
     ("Weapon", "?x")),

    ([("Missile", "?x"), ("Owns", "A", "?x")],
     ("Sells", "Robert", "?x", "A")),

    ([("American", "?x"),
      ("Weapon", "?y"),
      ("Sells", "?x", "?y", "?z"),
      ("Enemy", "?z", "America")],
     ("Criminal", "?x"))
]

def match_all(patterns, facts, subst=None):

    if subst is None:
        subst = {}

    if not patterns:
        return [subst]

    results = []

    first = apply_subst(patterns[0], subst)

    for fact in facts:
        new_subst = unify(first, fact, subst.copy())

        if new_subst is not None:
            results.extend(
                match_all(patterns[1:], facts, new_subst)
            )

    return results

print("UNIFICATION IN FIRST ORDER LOGIC")
print("Initial Facts:", facts)

changed = True

while changed:
    changed = False

    for premises, conclusion in rules:

        substitutions = match_all(premises, facts)

        for subst in substitutions:
            new_fact = apply_subst(conclusion, subst)

            if new_fact not in facts:
                facts.append(new_fact)
                print("Derived:", new_fact)
                changed = True
print("\nFinal Result:")

if ("Criminal", "Robert") in facts:
    print("Robert is a Criminal.")
else:
    print("Proof Failed.")


UNIFICATION IN FIRST ORDER LOGIC
Initial Facts: [('American', 'Robert'), ('Enemy', 'A', 'America'), ('Missile', 'M1'), ('Owns', 'A', 'M1')]
Derived: ('Weapon', 'M1')
Derived: ('Sells', 'Robert', 'M1', 'A')
Derived: ('Criminal', 'Robert')

Final Result:
Robert is a Criminal.
